In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CODE FOR TRAINING and SAVING MODEL

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# --- 1. CONFIGURATION ---
IMG_SIZE = (180, 180)
DATASET_PATH = '/content/drive/MyDrive/cs124p project/'
MODEL_SAVE_PATH = '/content/drive/MyDrive/cs124p/models/waste_classifier_model.h5'

# Define the categories
MY_CLASSES = ['Soda Can', 'Cardboard', 'Plastic Bottle', 'Glass Bottle']

def train_my_model():
    # 2. PREPARE DATA (With Data Augmentation)
    # This turns 1 photo into many by tilting and flipping them
    datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.3,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True
    )

    print("Loading images from Drive...")

    # Training Data
    train_gen = datagen.flow_from_directory(
        DATASET_PATH,
        target_size=IMG_SIZE,
        batch_size=32,
        class_mode='sparse',
        subset='training',
        classes=MY_CLASSES
    )

    # Validation Data (for testing while learning)
    val_gen = datagen.flow_from_directory(
        DATASET_PATH,
        target_size=IMG_SIZE,
        batch_size=32,
        class_mode='sparse',
        subset='validation',
        classes=MY_CLASSES
    )

    # 3. BUILD SMART MODEL (Transfer Learning)
    # We use MobileNetV2 because it's fast and already knows shapes/colors
    base_model = MobileNetV2(input_shape=(180, 180, 3), include_top=False, weights='imagenet')
    base_model.trainable = False # Freeze the pre-trained "brain"

    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dropout(0.2)(x) # Helps prevent the model from "memorizing"
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(len(MY_CLASSES), activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # 4. START TRAINING
    print(f"\nTraining started on {len(MY_CLASSES)} classes: {MY_CLASSES}")
    # We run for 10 "epochs" (10 full passes through your photos)
    model.fit(train_gen, validation_data=val_gen, epochs=10)

    # 5. SAVE THE KNOWLEDGE
    model.save(MODEL_SAVE_PATH)
    print(f"\nModel saved successfully.")

if os.path.exists(DATASET_PATH):
    train_my_model()
else:
    print(f"ERROR: The path {DATASET_PATH} was not found. Check your Drive folders!")

Loading images from Drive...
Found 110 images belonging to 4 classes.
Found 43 images belonging to 4 classes.


/tmp/ipykernel_4808/133364287.py:52: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(input_shape=(180, 180, 3), include_top=False, weights='imagenet')



Training started on 4 classes: ['Soda Can', 'Cardboard', 'Plastic Bottle', 'Glass Bottle']


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 31s 8s/step - accuracy: 0.4455 - loss: 1.4495 - val_accuracy: 0.5349 - val_loss: 0.9231
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 13s 4s/step - accuracy: 0.8455 - loss: 0.5514 - val_accuracy: 0.8837 - val_loss: 0.4348
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 17s 5s/step - accuracy: 0.9273 - loss: 0.2977 - val_accuracy: 0.8372 - val_loss: 0.5016
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 13s 4s/step - accuracy: 0.9636 - loss: 0.1826 - val_accuracy: 0.8837 - val_loss: 0.2976
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 12s 3s/step - accuracy: 0.9636 - loss: 0.1506 - val_accuracy: 0.8140 - val_loss: 0.3628
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 21s 3s/step - accuracy: 0.9636 - loss: 0.0916 - val_accuracy: 0.8605 - val_loss: 0.2721
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 13s 3s/step - accuracy: 0.9909 - loss: 0.0534 - val_accuracy: 0.8837 - val_loss: 0.3008
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 12s 3s/step - accuracy: 0.9909 - loss: 0.0375 - val_accuracy: 0.8372 - val_loss: 0.4047
Epoch 9/


Model saved successfully.


Trash-Scan

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
from IPython.display import clear_output

def predict_uploaded_image(image_bytes, model, class_names, img_size, output_widget):
    disposal_instructions = {
        'Soda Can': "Empty any remaining liquid, crush the can to save space, and place it in the Recyclables bin.",
        'Cardboard': "Flatten completely and ensure it is dry and free of food waste. Place in the Recyclables bin.",
        'Plastic Bottle': "Empty all liquid, remove the cap, crush the bottle, and dispose of it in the Recyclables bin.",
        'Glass Bottle': "Empty the contents and handle with care. Do not crush or break. Place gently in the Recyclables or designated Glass bin."
    }

    with output_widget: # Use the output_widget from the UI cell
        clear_output()
        try:
            # Convert uploaded bytes to image and preprocess
            img = Image.open(io.BytesIO(image_bytes)).resize(img_size)
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) / 255.0

            # Predict
            predictions = model.predict(img_array)
            score_index = np.argmax(predictions[0])
            confidence = 100 * np.max(predictions[0])
            result_label = class_names[score_index]

            instruction = disposal_instructions.get(result_label, "No specific disposal instruction found for this item.")

            # Display result
            fig, ax = plt.subplots(figsize=(6, 8)) # Adjust figure size to accommodate text

            ax.imshow(img)
            ax.set_title(f"Prediction: {result_label} ({confidence:.2f}%)")
            ax.axis('off')

            # Add instruction text below the image
            # Adjust y position to place text below the image area and ensure layout makes space
            fig.tight_layout(rect=[0, 0.1, 1, 1]) # Make space for text at the bottom (10% of figure height)
            ax.text(0.5, -0.15, f"Disposal Instruction:\n{instruction}",
                    transform=ax.transAxes, fontsize=20, va='top', ha='center', wrap=True)

            print(f"\nRESULT: {result_label} ({confidence:.2f}%)")
            print(f"Disposal Instruction: {instruction}")
            print()
            plt.show()
        except Exception as e:
            print(f"Error processing image: {e}")

In [ ]:
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import io
from PIL import Image
import tensorflow as tf
import matplotlib.pyplot as plt

# JAVASCRIPT TO ACCESS WEBCAM
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Scan Trash';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  return binary

# FUNCTION TO RUN WEBCAM SCAN
def run_webcam_scan(model, class_names, img_size, output_widget):
    disposal_instructions = {
        'Soda Can': "Empty any remaining liquid, crush the can to save space, and place it in the Recyclables bin.",
        'Cardboard': "Flatten completely and ensure it is dry and free of food waste. Place in the Recyclables bin.",
        'Plastic Bottle': "Empty all liquid, remove the cap, crush the bottle, and dispose of it in the Recyclables bin.",
        'Glass Bottle': "Empty the contents and handle with care. Do not crush or break. Place gently in the Recyclables or designated Glass bin."
    }

    with output_widget: # Use the output_widget from the UI cell
        clear_output()
        try:
            print("Point your camera at the trash and click 'Scan Trash'...")
            image_bytes = take_photo()

            # Convert captured photo to model format
            img = Image.open(io.BytesIO(image_bytes)).resize(img_size)
            img_array = tf.keras.preprocessing.image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0) / 255.0

            # Predict
            predictions = model.predict(img_array)
            result = class_names[np.argmax(predictions[0])]
            conf = np.max(predictions[0]) * 100

            instruction = disposal_instructions.get(result, "No specific disposal instruction found for this item.")

            # Display result
            fig, ax = plt.subplots(figsize=(6, 8)) # Adjust figure size to accommodate text

            ax.imshow(img)
            ax.set_title(f"Prediction: {result} ({conf:.2f}%)")
            ax.axis('off')

            # Add instruction text below the image
            fig.tight_layout(rect=[0, 0.1, 1, 1]) # Make space for text at the bottom
            ax.text(0.5, -0.15, f"Disposal Instruction:\n{instruction}",
                    transform=ax.transAxes, fontsize=20, va='top', ha='center', wrap=True)

            print(f"\nRESULT: {result} ({conf:.2f}%)")
            print(f"Disposal Instruction: {instruction}")
            print()
            plt.show()

        except Exception as err:
            print(f"Webcam scan failed: {str(err)}. Make sure you allowed camera access.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import tensorflow as tf

MODEL_PATH = '/content/drive/MyDrive/cs124p/models/waste_classifier_model.h5'
CLASS_NAMES = ['Soda Can', 'Cardboard', 'Plastic Bottle', 'Glass Bottle']
IMG_SIZE = (180, 180)

# Load Model
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print(f"Model '{MODEL_PATH}' loaded successfully for UI.")
except Exception as e:
    model = None
    print(f"Error loading model: {e}\nPlease ensure the model path is correct and the model exists.")

# UI Elements (used ipywidgets package)
output_widget = widgets.Output()

# Upload Button
upload_button = widgets.FileUpload(
    accept='image/*',  # .jpg only
    multiple=False  # Allow only one file to be uploaded at a time
)

# Webcam Button
webcam_button = widgets.Button(description='Scan with Webcam')

# Event Handlers
def on_upload_change(change):
    if model is None:
        with output_widget:
            clear_output()
            print("Model not loaded. Cannot process upload.")
        return
    if upload_button.value:
        # Get the first (and only) file's content
        uploaded_file = upload_button.value[tuple(upload_button.value.keys())[0]]
        image_bytes = uploaded_file['content']
        predict_uploaded_image(image_bytes, model, CLASS_NAMES, IMG_SIZE, output_widget)

def on_webcam_button_click(b):
    if model is None:
        with output_widget:
            clear_output()
            print("Model not loaded. Cannot start webcam scan.")
        return
    run_webcam_scan(model, CLASS_NAMES, IMG_SIZE, output_widget)

upload_button.observe(on_upload_change, names='value')
webcam_button.on_click(on_webcam_button_click)

# Display UI
display(widgets.VBox([
    widgets.HTML('<h2>Trash-Scan</h2>'),
    widgets.HTML('<p>Choose an option below to classify trash (Soda Can, Cardboard, Plastic Bottle, Glass Bottle):</p>'),
    widgets.HBox([upload_button, webcam_button]),
    output_widget
]))



Model '/content/drive/MyDrive/cs124p/models/waste_classifier_model.h5' loaded successfully for UI.
